# Generic streaming batch statistics

`cx.tl.batch_process` applies a mergeable statistic within experimental batches without loading the complete cell-by-gene matrix. This example computes a batch-corrected standard deviation for every perturbation and gene.

In [1]:
from dataclasses import replace
from pathlib import Path
import sys

import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp

ROOT = Path('../..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
import crispyx as cx

OUTPUT_DIR = ROOT / 'docs' / 'notebooks' / 'tutorial_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Create a batched example

The batches deliberately have different means and variances. Computing variability within each batch prevents shifts in batch means from inflating the result.

In [2]:
rng = np.random.default_rng(7)
rows, perturbations, batches = [], [], []
for batch_index, batch in enumerate(['lane_1', 'lane_2', 'lane_3']):
    for group_index, perturbation in enumerate(['control', 'KO_A', 'KO_B']):
        n_cells = 20 + 4 * batch_index + 2 * group_index
        block = rng.normal(
            loc=4 * batch_index + group_index,
            scale=0.5 + batch_index + 0.25 * group_index,
            size=(n_cells, 6),
        )
        rows.append(block)
        perturbations.extend([perturbation] * n_cells)
        batches.extend([batch] * n_cells)

X = np.vstack(rows)
obs = pd.DataFrame(
    {'perturbation': perturbations, 'batch': batches},
    index=[f'cell_{i}' for i in range(X.shape[0])],
)
var = pd.DataFrame(index=[f'gene_{i}' for i in range(X.shape[1])])
input_path = OUTPUT_DIR / 'batch_statistics_input.h5ad'
ad.AnnData(sp.csr_matrix(X), obs=obs, var=var).write(input_path)

## Define a streaming standard-deviation reducer

A reducer has three callbacks in per-group mode. `initialize` creates state for a gene chunk, `update` merges each cell chunk using the parallel-variance formula, and `finalize` returns the within-batch statistic plus its aggregation weight. Here the weight is the number of cells in that perturbation/batch.

In [3]:
def initialize_std(n_genes):
    return {'n': 0, 'mean': np.zeros(n_genes), 'm2': np.zeros(n_genes)}

def update_std(state, block):
    block_n = block.shape[0]
    block_mean = block.mean(axis=0)
    block_m2 = np.square(block - block_mean).sum(axis=0)
    if state['n'] == 0:
        state.update(n=block_n, mean=block_mean, m2=block_m2)
        return
    total_n = state['n'] + block_n
    delta = block_mean - state['mean']
    state['m2'] += block_m2 + delta**2 * state['n'] * block_n / total_n
    state['mean'] += delta * block_n / total_n
    state['n'] = total_n

def finalize_std(state):
    std = np.sqrt(state['m2'] / (state['n'] - 1))
    return cx.BatchStatistic(std, weight=state['n'])

std_reducer = cx.BatchReducer(
    initialize=initialize_std,
    update=update_std,
    finalize=finalize_std,
)

In [4]:
std_result = cx.tl.batch_process(
    input_path,
    std_reducer,
    groupby='perturbation',  # DE-compatible alias
    batch_column='batch',
    mode='group',
    statistic_name='std',
    chunk_size=3,       # genes per chunk
    cell_chunk_size=16, # cells per reducer update
    output_path=OUTPUT_DIR / 'batch_corrected_std.h5ad',
    force=True,
)

corrected_std = pd.DataFrame(
    std_result.backed.X[:],
    index=std_result.backed.obs_names,
    columns=std_result.backed.var_names,
)
corrected_std

,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5
perturbation,,,,,,
control,1.485919,1.653347,1.677128,1.442305,1.251690,1.413554
KO_A,1.661801,1.949438,1.499869,1.914997,1.692720,1.887972
KO_B,2.181261,2.000110,2.172170,2.193765,2.152863,2.456224


For group $g$ and gene $j$, the result is $\sum_b n_{gb}s_{gbj}/\sum_b n_{gb}$: a cell-count-weighted average of within-batch sample standard deviations. The between-batch mean shift is therefore excluded.

In [5]:
reference = []
for perturbation in corrected_std.index:
    batch_stds, batch_counts = [], []
    for batch in pd.unique(obs['batch']):
        mask = (obs['perturbation'] == perturbation) & (obs['batch'] == batch)
        batch_stds.append(X[mask].std(axis=0, ddof=1))
        batch_counts.append(mask.sum())
    reference.append(np.average(batch_stds, axis=0, weights=batch_counts))

np.testing.assert_allclose(corrected_std.to_numpy(), reference)
print('Streaming and direct calculations agree.')

Streaming and direct calculations agree.


## Comparison mode

For a group-versus-reference statistic, add a `compare(group_state, reference_state)` callback and call with `mode='comparison'`. The arguments `reference`/`control_label`, `groupby`/`perturbation_column`, and `perturbations` behave like crispyx DE functions. If no reference is supplied, crispyx infers the control label using the same rules as DE. Only batches containing both the group and reference contribute.

In [6]:
def compare_means(group_state, reference_state):
    n_group = group_state['n']
    n_reference = reference_state['n']
    return cx.BatchStatistic(
        group_state['mean'] - reference_state['mean'],
        weight=n_group * n_reference / (n_group + n_reference),
    )

# BatchReducer is a frozen dataclass, so attach `compare` with dataclasses.replace
# rather than mutating the reducer defined above.
contrast_reducer = replace(std_reducer, compare=compare_means)

contrast_result = cx.tl.batch_process(
    input_path,
    contrast_reducer,
    groupby='perturbation',
    reference='control',   # omit to let crispyx infer the control label
    batch_column='batch',
    mode='comparison',
    statistic_name='mean_difference',
    chunk_size=3,
    cell_chunk_size=16,
    output_path=OUTPUT_DIR / 'batch_corrected_mean_difference.h5ad',
    force=True,
)

mean_difference = pd.DataFrame(
    contrast_result.backed.X[:],
    index=contrast_result.backed.obs_names,
    columns=contrast_result.backed.var_names,
)
mean_difference

,gene_0,gene_1,gene_2,gene_3,gene_4,gene_5
perturbation,,,,,,
KO_A,0.806173,0.865520,1.386226,1.062358,0.845954,0.82975
KO_B,2.407209,1.858187,1.838216,2.538213,2.036634,1.94531


The reference group is not itself returned. For group $g$ and gene $j$ the result is $\sum_b w_{gb}(\bar{x}_{gbj} - \bar{x}_{rbj})/\sum_b w_{gb}$, where $w_{gb} = n_{gb}n_{rb}/(n_{gb} + n_{rb})$ is the harmonic weight of the group and reference cell counts in batch $b$. This is the same weighting the crispyx differential-expression functions use, and it gives batches with balanced group/reference representation the most influence.

Because the contrast is formed *within* each batch before averaging, the per-batch mean shifts built into the example data cancel out. The synthetic data was constructed so that `KO_A` sits one unit above `control` and `KO_B` two units above, in every batch. The recovered contrasts centre on 1 and 2 accordingly; the spread around those values is sampling noise from the modest number of cells per group and batch, not batch leakage.

In [7]:
reference_contrast = []
for perturbation in mean_difference.index:
    contrasts, weights = [], []
    for batch in pd.unique(obs['batch']):
        group_mask = (obs['perturbation'] == perturbation) & (obs['batch'] == batch)
        control_mask = (obs['perturbation'] == 'control') & (obs['batch'] == batch)
        n_group, n_control = group_mask.sum(), control_mask.sum()
        contrasts.append(X[group_mask].mean(axis=0) - X[control_mask].mean(axis=0))
        weights.append(n_group * n_control / (n_group + n_control))
    reference_contrast.append(np.average(contrasts, axis=0, weights=weights))

np.testing.assert_allclose(mean_difference.to_numpy(), reference_contrast)
print('Streaming and direct contrasts agree.')

Streaming and direct contrasts agree.


## Inspecting the result

`batch_process` writes a disk-backed AnnData. Alongside `X` it records the total aggregation weight per group and gene in a `weight_sum` layer, the number of batches that contributed to each group in `obs`, and the call parameters in `uns` — enough to tell a genuinely small effect apart from one estimated from very few cells.

In [8]:
backed = contrast_result.backed

print('result shape      :', backed.shape)
print('batches per group :', backed.obs['n_batches_used'].to_dict())
print('weight_sum (gene_0):', dict(zip(backed.obs_names, backed.layers['weight_sum'][:, 0])))
print()
for key in ('statistic_name', 'mode', 'perturbation_column', 'batch_column'):
    if key in backed.uns:
        print(f'uns[{key!r}] = {backed.uns[key]!r}')

result shape      : (2, 6)
batches per group : {'KO_A': 3, 'KO_B': 3}
weight_sum (gene_0): {'KO_A': np.float64(37.43894909688014), 'KO_B': np.float64(38.76550116550116)}

uns['statistic_name'] = 'mean_difference'
uns['mode'] = 'comparison'
uns['perturbation_column'] = 'perturbation'
uns['batch_column'] = 'batch'
